<a href="https://colab.research.google.com/github/sena-atadja/Stanbic-Bank-Credit-Risk-Model/blob/main/Stanbic_Bank_Ghana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
import numpy as np

file_name = "ds-sme_loan_applications_stanbic_gh.csv"

df_raw = pd.read_csv(
    file_name,
    encoding="utf-8-sig",
    dtype="string"
)

df = df_raw.copy()

df.shape



Saving ds-sme_loan_applications_stanbic_gh.csv to ds-sme_loan_applications_stanbic_gh.csv


(3037, 29)

In [ ]:
df.head()

,application_id,business_name,sector,region,ethnic_group,years_in_operation,owner_gender,owner_age,disability_status,num_employees,...,collateral_value_ghs,previous_loan_count,previous_default,credit_bureau_score,rm_recommendation,internal_risk_grade,application_date,contact_phone,days_past_due_current,loan_default
0,SBG-2025-01722,Achimota Farms,Fishing,Ashanti,Akan,0.6,Female,34,No,2,...,29174.78,0,<NA>,<NA>,Approve,A,05/07/2024,0558707981,0,0
1,SBG-2025-02335,Kofi MotorsCo,Hospitality,Savannah,Mole-Dagbani,0.6,Female,31,No,19,...,15816.11,2,Yes,628,Refer,B,06/07/2025,0270660120,0,0
2,SBG-2025-00444,Dansoman TradingLtd,Hospitality,Greater Accra,Ga-Adangbe,2.3,female,32,No,8,...,13374.67,1,No,371,Decline,C,24/10/2025,0546332387,0,0
3,SBG-2025-02333,Airport City ConsultingCo,Manufacturing,Ashanti,Akan,4.6,male,56,No,5,...,60667.51,1,No,<NA>,Approve,B,18/10/2024,0244150848,0,0
4,SBG-2025-02884,Grace Foods LtdEnterprises,Education,Ashanti,Akan,0.4,Male,39,No,4,...,<NA>,4,No,<NA>,Refer,C,02/20/2025,0246443276,0,0


In [ ]:
df.info()

missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3037 entries, 0 to 3036
Data columns (total 29 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   application_id                3037 non-null   string
 1   business_name                 3033 non-null   string
 2   sector                        3037 non-null   string
 3   region                        3037 non-null   string
 4   ethnic_group                  3037 non-null   string
 5   years_in_operation            3037 non-null   string
 6   owner_gender                  3037 non-null   string
 7   owner_age                     3037 non-null   string
 8   disability_status             3037 non-null   string
 9   num_employees                 3037 non-null   string
 10  annual_revenue_ghs            2950 non-null   string
 11  monthly_momo_volume_ghs       2583 non-null   string
 12  avg_monthly_bank_balance_ghs  2728 non-null   string
 13  bank_account_tenur

,0
credit_bureau_score,1967
gra_tin,1031
previous_default,604
monthly_momo_volume_ghs,454
collateral_value_ghs,389
collateral_type,367
avg_monthly_bank_balance_ghs,309
annual_revenue_ghs,87
business_name,4
ethnic_group,0


In [ ]:
print("Rows before:", len(df))

df = df.drop_duplicates().copy()

print("Rows after:", len(df))


Rows before: 3037
Rows after: 3000


In [ ]:
df["owner_gender"] = (
    df["owner_gender"]
    .str.strip()
    .str.lower()
    .map({
        "male": "Male",
        "m": "Male",
        "female": "Female",
        "f": "Female"
    })
)

df["has_momo_account"] = (
    df["has_momo_account"]
    .str.strip()
    .str.lower()
    .map({
        "yes": "Yes",
        "y": "Yes",
        "1": "Yes",
        "true": "Yes",
        "no": "No",
        "n": "No",
        "0": "No",
        "false": "No"
    })
)

df[["owner_gender", "has_momo_account"]].head()


,owner_gender,has_momo_account
0,Female,Yes
1,Female,Yes
2,Female,Yes
3,Male,No
4,Male,Yes


In [ ]:
df = df.drop(
    columns=["ethnic_group", "disability_status", "rm_recommendation", "internal_risk_grade", "days_past_due_current"],
    errors="ignore"
)

df.head()

df.to_csv("sme_loans_cleaned.csv", index=False)

files.download("sme_loans_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Check duplicate application IDs before cleaning
duplicate_app_ids = df[df["application_id"].duplicated(keep=False)]

duplicate_app_ids.sort_values("application_id").head(20)
print("Rows with duplicate application_id:", duplicate_app_ids.shape[0])
print("Exact duplicate rows:", df.duplicated().sum())


Rows with duplicate application_id: 0
Exact duplicate rows: 0


In [ ]:
# DQ-02: Credit bureau score missingness

df["credit_bureau_score"] = pd.to_numeric(
    df["credit_bureau_score"],
    errors="coerce"
)

df["credit_bureau_missing_flag"] = df["credit_bureau_score"].isna()

df["credit_bureau_status"] = np.where(
    df["credit_bureau_missing_flag"],
    "No bureau record / missing",
    "Bureau score available"
)

df["credit_bureau_status"].value_counts(dropna=False)

missing_count = df["credit_bureau_missing_flag"].sum()
missing_rate = df["credit_bureau_missing_flag"].mean()

# Convert raw bureau score to numeric
df["credit_bureau_score"] = pd.to_numeric(
    df["credit_bureau_score"],
    errors="coerce"
)


print("Missing bureau scores:", missing_count)
print("Missing rate:", round(missing_rate * 100, 2), "%")

df["credit_bureau_score"] = pd.to_numeric(
    df["credit_bureau_score"],
    errors="coerce"
)

df["credit_bureau_missing_flag"] = df["credit_bureau_score"].isna()

df["credit_bureau_status"] = np.where(
    df["credit_bureau_missing_flag"],
    "No bureau record",
    "Bureau score available"
)

df["credit_bureau_score_clean"] = df["credit_bureau_score"]

df.loc[
    df["credit_bureau_score_clean"].lt(300) |
    df["credit_bureau_score_clean"].gt(850),
    "credit_bureau_score_clean"
] = np.nan

df["credit_bureau_out_of_range_flag"] = (
    df["credit_bureau_score"].lt(300) |
    df["credit_bureau_score"].gt(850)
)


def bureau_band(row):
    if pd.isna(row["credit_bureau_score"]):
        return "No bureau record"
    elif row["credit_bureau_score"] < 300:
        return "Invalid / below range"
    elif row["credit_bureau_score"] < 500:
        return "High risk"
    elif row["credit_bureau_score"] < 650:
        return "Medium risk"
    elif row["credit_bureau_score"] <= 850:
        return "Low risk"
    else:
        return "Invalid / above range"

df["credit_bureau_band"] = df.apply(bureau_band, axis=1)

df["credit_bureau_band"].value_counts(dropna=False)

# DQ-02: Credit bureau score missingness and out-of-range handling

df["credit_bureau_score"] = pd.to_numeric(
    df["credit_bureau_score"],
    errors="coerce"
)

# Missing bureau history is meaningful, so preserve it explicitly
df["credit_bureau_missing_flag"] = df["credit_bureau_score"].isna()

# Flag invalid/out-of-range bureau scores
df["credit_bureau_out_of_range_flag"] = (
    df["credit_bureau_score"].lt(300) |
    df["credit_bureau_score"].gt(850)
)

# Keep only valid bureau scores in the clean field
df["credit_bureau_score_clean"] = df["credit_bureau_score"]

df.loc[
    df["credit_bureau_missing_flag"] |
    df["credit_bureau_out_of_range_flag"],
    "credit_bureau_score_clean"
] = np.nan

# Use sentinel value for modelling
# -1 means no usable bureau score
df["credit_bureau_score_model"] = df["credit_bureau_score_clean"].fillna(-1)

# Explainable status field
df["credit_bureau_status"] = np.select(
    [
        df["credit_bureau_missing_flag"],
        df["credit_bureau_out_of_range_flag"]
    ],
    [
        "No bureau record",
        "Invalid bureau score"
    ],
    default="Valid bureau score"
)

# Explainable risk band
def bureau_band(row):
    if row["credit_bureau_missing_flag"]:
        return "No bureau record"
    elif row["credit_bureau_out_of_range_flag"]:
        return "Invalid bureau score"
    elif row["credit_bureau_score_clean"] < 500:
        return "High risk"
    elif row["credit_bureau_score_clean"] < 650:
        return "Medium risk"
    else:
        return "Low risk"

df["credit_bureau_band"] = df.apply(bureau_band, axis=1)

missing_count = df["credit_bureau_missing_flag"].sum()
missing_rate = df["credit_bureau_missing_flag"].mean()

print("Missing bureau scores:", missing_count)
print("Missing rate:", round(missing_rate * 100, 2), "%")

df[
    [
        "credit_bureau_score",
        "credit_bureau_score_clean",
        "credit_bureau_score_model",
        "credit_bureau_missing_flag",
        "credit_bureau_out_of_range_flag",
        "credit_bureau_status",
        "credit_bureau_band"
    ]
].head()


# DQ-03: Previous default missingness

df["previous_loan_count"] = pd.to_numeric(
    df["previous_loan_count"],
    errors="coerce"
)

df["previous_default_raw"] = df["previous_default"]

df["previous_default_clean"] = (
    df["previous_default"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({
        "yes": "Yes",
        "no": "No"
    })
)

df["previous_default_status"] = np.select(
    [
        df["previous_loan_count"].eq(0) & df["previous_default_clean"].isna(),
        df["previous_loan_count"].gt(0) & df["previous_default_clean"].isna(),
        df["previous_default_clean"].eq("Yes"),
        df["previous_default_clean"].eq("No")
    ],
    [
        "No prior loan history",
        "Missing despite prior loan history",
        "Prior default",
        "No prior default"
    ],
    default="Review"
)

df["previous_default_status"].value_counts(dropna=False)

df["no_prior_loan_history_flag"] = (
    df["previous_default_status"] == "No prior loan history"
)

df["previous_default_missing_with_prior_history_flag"] = (
    df["previous_default_status"] == "Missing despite prior loan history"
)

df["prior_default_flag"] = (
    df["previous_default_status"] == "Prior default"
)

df["previous_default_model"] = df["previous_default_status"].replace({
    "No prior loan history": "Not applicable",
    "Missing despite prior loan history": "Unknown",
    "Prior default": "Yes",
    "No prior default": "No",
    "Review": "Unknown"
})

pd.crosstab(
    df["previous_loan_count"],
    df["previous_default_status"],
    dropna=False
)



Missing bureau scores: 1939
Missing rate: 64.63 %


TypeError: invalid entry 1 in condlist: should be boolean ndarray

# Data Quality Cleaning Summary

This notebook prepares the Stanbic Bank Ghana SME loan applications dataset for analysis and future credit decision modelling. The raw data is preserved in `df_raw`, while cleaning and feature preparation are performed on a working copy called `df`.

## Completed Cleaning Steps

### DQ-01: Duplicate Application IDs
**Issue:** Some `application_id` values appeared more than once, which could cause double-counting and corrupt model training.

**Action:** Exact duplicate records were removed using `drop_duplicates()` before analysis.

**Treatment:**  
- Dropped exact duplicate rows.
- Confirmed that `application_id` is unique after deduplication.
- Any future non-identical duplicate IDs should be flagged for manual review.

---

### DQ-02: Credit Bureau Score Missingness
**Issue:** `credit_bureau_score` is missing for a large share of applicants. This likely represents applicants with no credit bureau record or thin credit files.

**Action:** Missing bureau scores were not dropped and were not blindly imputed as average scores.

**Treatment:**  
- Converted `credit_bureau_score` to numeric.
- Created `credit_bureau_missing_flag`.
- Created `credit_bureau_status`.
- Created `credit_bureau_band` for explainability.
- Missing bureau score is treated as “No bureau record / missing,” not as good or bad credit by default.

**Policy note:**  
Applicants without bureau records should not be automatically declined. For SME lending, missing bureau history may indicate informal or first-time borrowers and should trigger alternative data review or referral.

---

### DQ-03: Previous Default Missingness
**Issue:** `previous_default` is missing for applicants with no prior loan history. It must not be imputed as `"No"` without validation.

**Action:** Missing `previous_default` values were interpreted using `previous_loan_count`.

**Treatment:**  
- Converted `previous_loan_count` to numeric.
- Created `previous_default_status`.
- Created `no_prior_loan_history_flag`.
- Created `prior_default_flag`.
- Created `previous_default_model`.

**Policy note:**  
For applicants with `previous_loan_count = 0`, `previous_default` is treated as “No prior loan history / Not applicable,” whether the raw value was blank or `"No"`. This avoids incorrectly treating thin-file applicants as proven good borrowers.

---

## Recommended Remaining Cleaning Actions

### DQ-04: Sensitive Attribute Exclusion
Remove `ethnic_group` and `disability_status` from the modelling dataset.

**Reason:** These are sensitive/protected attributes. They should not be used directly in automated credit decisions. They may be retained separately only for fairness monitoring if governance allows.

---

### DQ-05: Owner Gender Encoding
Normalize `owner_gender` values into consistent categories.

**Recommended treatment:**  
Map `M`, `male`, and `Male` to `Male`; map `F`, `female`, and `Female` to `Female`.

---

### DQ-06: MoMo Account Encoding
Normalize `has_momo_account` into a binary variable.

**Recommended treatment:**  
Map `Yes`, `yes`, `Y`, `1`, and `TRUE` to `1`; map `No`, `no`, `N`, `0`, and `FALSE` to `0`.

---

### DQ-07: Annual Revenue Currency and Formatting
Clean `annual_revenue_ghs`, which contains blanks, plain numbers, `GHS` prefixes, commas, and dollar-prefixed values.

**Recommended treatment:**  
- Remove `GHS` and commas.
- Convert clean GHS values to numeric.
- Flag `$` values for review instead of automatically treating them as GHS.
- Create `annual_revenue_clean_ghs`.
- Create `annual_revenue_usd_flag`.

---

### DQ-08: Mixed Date Formats
`application_date` appears to contain mixed `DD/MM/YYYY` and `MM/DD/YYYY` formats.

**Recommended treatment:**  
- Identify likely date format per record.
- Standardize to `YYYY-MM-DD`.
- Flag ambiguous dates for review.
- Avoid using `application_date` for time-based validation until resolved.

---

### DQ-09: Phone Number Format
Ensure `contact_phone` is stored as text and retains leading zeroes.

**Recommended treatment:**  
- Cast phone numbers to string.
- Remove non-digit characters.
- Add a leading `0` where a valid 9-digit Ghana number lost it.
- Create `phone_valid_flag`.

---

### DQ-10: GRA TIN Format
Validate `gra_tin` and create a usable flag.

**Recommended treatment:**  
- Standardize to uppercase text.
- Treat `PENDING` as pending/missing, not valid.
- Create `has_valid_tin`.
- Create `gra_tin_pending_flag`.

---

### DQ-11: Owner Age and Years in Operation
Some values are impossible or implausible.

**Recommended treatment:**  
- Set owner ages below 18 or above 80 to missing.
- Flag invalid owner ages.
- Set negative or extreme `years_in_operation` values to missing.
- Create review flags before modelling.

---

### DQ-12: MoMo Inconsistency
Some applicants marked as having no MoMo account have positive MoMo volume.

**Recommended treatment:**  
- Create `has_momo_from_volume`.
- Create `momo_account_volume_conflict_flag`.
- For modelling, prefer transaction evidence where available.

---

### DQ-13: Collateral Mismatches
Some secured collateral types have missing collateral values.

**Recommended treatment:**  
- Create `has_collateral`.
- Create `collateral_missing_value_flag`.
- Keep the record but refer for review where collateral value is missing.

---

### DQ-14: Credit Bureau Out-of-Range Values
Some bureau scores are outside expected range.

**Recommended treatment:**  
- Set scores below 300 or above 850 to missing.
- Create out-of-range flags.
- Do not cap silently without confirming bureau score scale.

---

### DQ-15: Loan-to-Revenue Extremes
Some requested loan amounts are very high relative to annual revenue.

**Recommended treatment:**  
- Create `loan_to_revenue_ratio`.
- Flag records where requested loan is greater than 5x annual revenue.
- Use this as a review/referral signal, not necessarily an automatic decline.

---

## Modelling Principle

The cleaned dataset should distinguish between:

1. **Automatically cleaned values**  
   Examples: duplicate removal, category normalization, numeric parsing.

2. **Missing but meaningful values**  
   Examples: no bureau record, no prior loan history, missing TIN.

3. **Records requiring human review**  
   Examples: USD revenue values, ambiguous dates, collateral mismatches, impossible ages, high loan-to-revenue ratios.

This is important because the final prototype will output `Approve`, `Decline`, or `Refer to Human`. Many data quality issues should naturally feed into the `Refer` decision rather than being silently imputed.


In [ ]:
import pandas as pd
import numpy as np

# --------------------------------------------------
# DQ-04: Remove sensitive attributes from modelling data
# --------------------------------------------------

df = df.drop(
    columns=["ethnic_group", "disability_status"],
    errors="ignore"
)

# --------------------------------------------------
# DQ-05: Owner gender encoding
# --------------------------------------------------

df["owner_gender_raw"] = df["owner_gender"]

df["owner_gender_clean"] = (
    df["owner_gender"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({
        "male": "Male",
        "m": "Male",
        "female": "Female",
        "f": "Female"
    })
)

df["owner_gender_unmapped_flag"] = df["owner_gender_clean"].isna()

# --------------------------------------------------
# DQ-06: MoMo account encoding
# --------------------------------------------------

df["has_momo_account_raw"] = df["has_momo_account"]

df["has_momo_account_clean"] = (
    df["has_momo_account"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({
        "yes": 1,
        "y": 1,
        "1": 1,
        "true": 1,
        "no": 0,
        "n": 0,
        "0": 0,
        "false": 0
    })
)

df["has_momo_account_unmapped_flag"] = df["has_momo_account_clean"].isna()

# --------------------------------------------------
# Helper for numeric conversion
# --------------------------------------------------

def clean_numeric(series):
    return pd.to_numeric(
        series.astype("string")
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

# --------------------------------------------------
# DQ-07: Annual revenue currency and formatting
# --------------------------------------------------

df["annual_revenue_raw"] = df["annual_revenue_ghs"]

annual_raw = df["annual_revenue_ghs"].astype("string").str.strip()

df["annual_revenue_usd_flag"] = annual_raw.str.contains(r"\$", na=False)
df["annual_revenue_ghs_prefix_flag"] = annual_raw.str.contains("GHS", case=False, na=False)

annual_clean_text = (
    annual_raw
    .str.replace("GHS", "", case=False, regex=False)
    .str.replace(",", "", regex=False)
    .str.replace("$", "", regex=False)
    .str.strip()
)

df["annual_revenue_clean_ghs"] = pd.to_numeric(
    annual_clean_text,
    errors="coerce"
)

# Do not silently treat dollar values as GHS
df.loc[df["annual_revenue_usd_flag"], "annual_revenue_clean_ghs"] = np.nan

# --------------------------------------------------
# Convert other numeric columns needed below
# --------------------------------------------------

numeric_columns = [
    "monthly_momo_volume_ghs",
    "loan_amount_requested_ghs",
    "collateral_value_ghs",
    "owner_age",
    "years_in_operation",
    "credit_bureau_score"
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = clean_numeric(df[col])

# --------------------------------------------------
# DQ-08: Mixed date formats
# --------------------------------------------------

df["application_date_raw"] = df["application_date"]

date_raw = df["application_date"].astype("string").str.strip()
date_parts = date_raw.str.extract(r"^(\d{1,2})/(\d{1,2})/(\d{4})$")

first = pd.to_numeric(date_parts[0], errors="coerce")
second = pd.to_numeric(date_parts[1], errors="coerce")

df["application_date_format_flag"] = "Invalid date pattern"

df.loc[
    first.gt(12) & second.le(12),
    "application_date_format_flag"
] = "Likely DD/MM/YYYY"

df.loc[
    first.le(12) & second.gt(12),
    "application_date_format_flag"
] = "Likely MM/DD/YYYY"

df.loc[
    first.le(12) & second.le(12),
    "application_date_format_flag"
] = "Ambiguous DD/MM vs MM/DD"

parsed_ddmmyyyy = pd.to_datetime(date_raw, format="%d/%m/%Y", errors="coerce")
parsed_mmddyyyy = pd.to_datetime(date_raw, format="%m/%d/%Y", errors="coerce")

df["application_date_clean"] = parsed_ddmmyyyy

df.loc[
    df["application_date_format_flag"].eq("Likely MM/DD/YYYY"),
    "application_date_clean"
] = parsed_mmddyyyy

df["application_date_needs_review_flag"] = df["application_date_format_flag"].isin([
    "Ambiguous DD/MM vs MM/DD",
    "Invalid date pattern"
])

# --------------------------------------------------
# DQ-09: Phone number format
# --------------------------------------------------

df["contact_phone_raw"] = df["contact_phone"]

df["contact_phone"] = (
    df["contact_phone"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"\D", "", regex=True)
    .str.strip()
)

df["phone_leading_zero_added_flag"] = (
    df["contact_phone"].str.len().eq(9) &
    ~df["contact_phone"].str.startswith("0")
)

df.loc[
    df["phone_leading_zero_added_flag"],
    "contact_phone"
] = "0" + df.loc[df["phone_leading_zero_added_flag"], "contact_phone"]

df["phone_valid_flag"] = df["contact_phone"].str.match(r"^0\d{9}$", na=False)

# --------------------------------------------------
# DQ-10: GRA TIN format
# --------------------------------------------------

df["gra_tin_raw"] = df["gra_tin"]

df["gra_tin"] = (
    df["gra_tin"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df["gra_tin_pending_flag"] = df["gra_tin"].eq("PENDING")
df["has_valid_tin"] = df["gra_tin"].str.match(r"^[A-Z]\d{9}$", na=False)

# --------------------------------------------------
# DQ-11: Owner age and years in operation
# --------------------------------------------------

df["owner_age_invalid_flag"] = (
    df["owner_age"].lt(18) |
    df["owner_age"].gt(80) |
    df["owner_age"].isna()
)

df["owner_age_clean"] = df["owner_age"]

df.loc[
    df["owner_age_invalid_flag"],
    "owner_age_clean"
] = np.nan

df["years_in_operation_invalid_flag"] = (
    df["years_in_operation"].lt(0) |
    df["years_in_operation"].gt(50) |
    df["years_in_operation"].isna()
)

df["years_in_operation_clean"] = df["years_in_operation"].clip(lower=0, upper=50)

df.loc[
    df["years_in_operation_invalid_flag"],
    "years_in_operation_clean"
] = np.nan

# --------------------------------------------------
# DQ-12: MoMo inconsistency
# --------------------------------------------------

df["has_momo_from_volume"] = np.where(
    df["monthly_momo_volume_ghs"].fillna(0).gt(0),
    1,
    0
)

df["momo_account_volume_conflict_flag"] = (
    df["has_momo_account_clean"].notna() &
    df["has_momo_account_clean"].ne(df["has_momo_from_volume"])
)

# For modelling, prefer transaction evidence
df["has_momo_final"] = df["has_momo_from_volume"]

# --------------------------------------------------
# DQ-13: Collateral mismatches
# --------------------------------------------------

secured_collateral_types = [
    "Property",
    "Land",
    "Vehicle",
    "Equipment",
    "Stock/Inventory",
    "Cash Deposit"
]

unsecured_collateral_types = [
    "None",
    "Guarantor Only"
]

df["has_collateral"] = np.where(
    df["collateral_type"].isin(secured_collateral_types),
    1,
    0
)

df["collateral_missing_value_flag"] = (
    df["collateral_type"].isin(secured_collateral_types) &
    df["collateral_value_ghs"].isna()
)

df["collateral_value_without_collateral_flag"] = (
    df["collateral_type"].isin(unsecured_collateral_types) &
    df["collateral_value_ghs"].fillna(0).gt(0)
)

df["collateral_mismatch_flag"] = (
    df["collateral_missing_value_flag"] |
    df["collateral_value_without_collateral_flag"]
)

# --------------------------------------------------
# DQ-14: Credit bureau out-of-range values
# --------------------------------------------------

df["credit_bureau_below_300_flag"] = df["credit_bureau_score"].lt(300)
df["credit_bureau_above_850_flag"] = df["credit_bureau_score"].gt(850)

df["credit_bureau_score_clean"] = df["credit_bureau_score"]

df.loc[
    df["credit_bureau_below_300_flag"] |
    df["credit_bureau_above_850_flag"],
    "credit_bureau_score_clean"
] = np.nan

# Preserve your existing missing-status logic if already created
if "credit_bureau_status" not in df.columns:
    df["credit_bureau_status"] = "Valid bureau score"

df.loc[
    df["credit_bureau_score"].isna(),
    "credit_bureau_status"
] = "No bureau record / missing"

df.loc[
    df["credit_bureau_below_300_flag"],
    "credit_bureau_status"
] = "Invalid below range"

df.loc[
    df["credit_bureau_above_850_flag"],
    "credit_bureau_status"
] = "Invalid above range"

# --------------------------------------------------
# DQ-15: Loan-to-revenue extremes
# --------------------------------------------------

df["loan_to_revenue_ratio"] = (
    df["loan_amount_requested_ghs"] /
    df["annual_revenue_clean_ghs"]
)

df.loc[
    df["annual_revenue_clean_ghs"].isna() |
    df["annual_revenue_clean_ghs"].le(0),
    "loan_to_revenue_ratio"
] = np.nan

df["loan_to_revenue_extreme_flag"] = df["loan_to_revenue_ratio"].gt(5)

# --------------------------------------------------
# Overall data quality review flag
# --------------------------------------------------

review_flag_columns = [
    "owner_gender_unmapped_flag",
    "has_momo_account_unmapped_flag",
    "annual_revenue_usd_flag",
    "application_date_needs_review_flag",
    "phone_leading_zero_added_flag",
    "gra_tin_pending_flag",
    "owner_age_invalid_flag",
    "years_in_operation_invalid_flag",
    "momo_account_volume_conflict_flag",
    "collateral_mismatch_flag",
    "credit_bureau_below_300_flag",
    "credit_bureau_above_850_flag",
    "loan_to_revenue_extreme_flag"
]

df["any_data_quality_review_flag"] = (
    df[review_flag_columns]
    .fillna(False)
    .astype(bool)
    .any(axis=1)
)

print("Remaining data quality cleaning complete.")
print("Current dataset shape:", df.shape)

dq_summary = pd.Series({
    "owner_gender_unmapped": df["owner_gender_unmapped_flag"].sum(),
    "momo_encoding_unmapped": df["has_momo_account_unmapped_flag"].sum(),
    "annual_revenue_usd_escalate": df["annual_revenue_usd_flag"].sum(),
    "dates_needing_review": df["application_date_needs_review_flag"].sum(),
    "invalid_phone_numbers": (~df["phone_valid_flag"]).sum(),
    "pending_tin": df["gra_tin_pending_flag"].sum(),
    "valid_tin": df["has_valid_tin"].sum(),
    "invalid_owner_age": df["owner_age_invalid_flag"].sum(),
    "invalid_years_in_operation": df["years_in_operation_invalid_flag"].sum(),
    "momo_conflicts": df["momo_account_volume_conflict_flag"].sum(),
    "collateral_mismatches": df["collateral_mismatch_flag"].sum(),
    "bureau_below_300": df["credit_bureau_below_300_flag"].sum(),
    "bureau_above_850": df["credit_bureau_above_850_flag"].sum(),
    "loan_to_revenue_gt_5x": df["loan_to_revenue_extreme_flag"].sum(),
    "records_needing_any_review": df["any_data_quality_review_flag"].sum()
})

dq_summary


Remaining data quality cleaning complete.
Current dataset shape: (3000, 66)


,0
owner_gender_unmapped,0
momo_encoding_unmapped,0
annual_revenue_usd_escalate,151
dates_needing_review,1267
invalid_phone_numbers,0
pending_tin,106
valid_tin,1875
invalid_owner_age,41
invalid_years_in_operation,9
momo_conflicts,1164


Preprocessing Model

In [ ]:
SENSITIVE_COLUMNS = ["ethnic_group", "disability_status", "rm_recommendations", "internal_risk_grade", "days_past_due_current"]

CATEGORICAL_COLUMNS = ["sector", "region", "loan_purpose", "collateral_type"]

SECURED_COLLATERAL_TYPES = [
    "Property", "Land", "Vehicle", "Equipment",
    "Stock/Inventory", "Cash Deposit"
]

UNSECURED_COLLATERAL_TYPES = [
    "None", "Guarantor Only"
]

REQUIRED_COLUMNS = [
    "application_id", "business_name", "sector", "region",
    "ethnic_group", "disability_status",
    "years_in_operation", "owner_gender", "owner_age",
    "num_employees", "annual_revenue_ghs",
    "monthly_momo_volume_ghs", "avg_monthly_bank_balance_ghs",
    "bank_account_tenure_months", "has_momo_account", "gra_tin",
    "loan_amount_requested_ghs", "loan_purpose", "collateral_type",
    "collateral_value_ghs", "previous_loan_count", "previous_default",
    "credit_bureau_score", "rm_recommendation", "internal_risk_grade",
    "application_date", "contact_phone", "days_past_due_current",
    "loan_default"
]


def clean_numeric(series):
    return pd.to_numeric(
        series.astype("string")
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )


def preprocess_applications(raw_data):
    """
    Reusable preprocessing function for both:
    1. historical training data
    2. one new raw application at inference time

    Input:
        raw_data: pandas DataFrame or dictionary

    Output:
        cleaned dataframe, data quality log dataframe
    """

    if isinstance(raw_data, dict):
        df = pd.DataFrame([raw_data])
    else:
        df = raw_data.copy()

    for col in REQUIRED_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA

    dq_log = []

    def log_issue(issue_code, severity, field, handling):
        dq_log.append({
            "issue_code": issue_code,
            "severity": severity,
            "field": field,
            "handling": handling
        })

    # Preserve raw values except sensitive attributes
    for col in df.columns:
        if col not in SENSITIVE_COLUMNS:
            df[f"{col}_raw"] = df[col]

    # DQ-00: Remove sensitive attributes
    df = df.drop(columns=SENSITIVE_COLUMNS, errors="ignore")

    log_issue(
        "DQ-00",
        "Governance",
        "ethnic_group, disability_status",
        "Removed from modelling dataset. May be retained separately only for fairness monitoring."
    )

    # DQ-01: Duplicate application IDs
    if len(df) > 1:
        before = len(df)
        df = df.drop_duplicates().copy()
        after = len(df)

        df["duplicate_application_id_flag"] = df["application_id"].duplicated(keep=False)

        log_issue(
            "DQ-01",
            "Critical",
            "application_id",
            f"Removed {before - after} exact duplicate rows. Remaining duplicate IDs flagged for review."
        )
    else:
        df["duplicate_application_id_flag"] = False

    # DQ-02: Annual revenue currency and formatting
    annual_raw = df["annual_revenue_ghs"].astype("string").str.strip()

    df["annual_revenue_usd_flag"] = annual_raw.str.contains(r"\$", na=False)
    df["annual_revenue_ghs_prefix_flag"] = annual_raw.str.contains("GHS", case=False, na=False)

    annual_clean_text = (
        annual_raw
        .str.replace("GHS", "", case=False, regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip()
    )

    df["annual_revenue_clean_ghs"] = pd.to_numeric(
        annual_clean_text,
        errors="coerce"
    )

    df.loc[df["annual_revenue_usd_flag"], "annual_revenue_clean_ghs"] = np.nan

    log_issue(
        "DQ-02",
        "High",
        "annual_revenue_ghs",
        "Parsed GHS values. Dollar-denominated values flagged and set to missing pending review."
    )

    # DQ-03: Mixed date formats
    date_raw = df["application_date"].astype("string").str.strip()

    date_parts = date_raw.str.extract(r"^(\d{1,2})/(\d{1,2})/(\d{4})$")
    first = pd.to_numeric(date_parts[0], errors="coerce")
    second = pd.to_numeric(date_parts[1], errors="coerce")

    df["application_date_format_flag"] = "Invalid date pattern"

    df.loc[first.gt(12) & second.le(12), "application_date_format_flag"] = "Likely DD/MM/YYYY"
    df.loc[first.le(12) & second.gt(12), "application_date_format_flag"] = "Likely MM/DD/YYYY"
    df.loc[first.le(12) & second.le(12), "application_date_format_flag"] = "Ambiguous DD/MM vs MM/DD"

    parsed_ddmmyyyy = pd.to_datetime(date_raw, format="%d/%m/%Y", errors="coerce")
    parsed_mmddyyyy = pd.to_datetime(date_raw, format="%m/%d/%Y", errors="coerce")

    df["application_date_clean"] = parsed_ddmmyyyy

    df.loc[
        df["application_date_format_flag"].eq("Likely MM/DD/YYYY"),
        "application_date_clean"
    ] = parsed_mmddyyyy

    df["application_date_needs_review_flag"] = df["application_date_format_flag"].isin([
        "Ambiguous DD/MM vs MM/DD",
        "Invalid date pattern"
    ])

    log_issue(
        "DQ-03",
        "High",
        "application_date",
        "Parsed likely date formats. Ambiguous or invalid dates flagged for review."
    )

    # DQ-04: Phone numbers
    df["contact_phone"] = (
        df["contact_phone"]
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"\D", "", regex=True)
        .str.strip()
    )

    df["phone_leading_zero_added_flag"] = (
        df["contact_phone"].str.len().eq(9) &
        ~df["contact_phone"].str.startswith("0")
    )

    df.loc[
        df["phone_leading_zero_added_flag"],
        "contact_phone"
    ] = "0" + df.loc[df["phone_leading_zero_added_flag"], "contact_phone"]

    df["phone_valid_flag"] = df["contact_phone"].str.match(r"^0\d{9}$", na=False)

    log_issue(
        "DQ-04",
        "Medium",
        "contact_phone",
        "Stored as string, removed non-digits, restored missing leading zero, and created validity flag."
    )

    # DQ-05: GRA TIN
    df["gra_tin"] = df["gra_tin"].astype("string").str.strip().str.upper()

    df["gra_tin_pending_flag"] = df["gra_tin"].eq("PENDING")
    df["has_valid_tin"] = df["gra_tin"].str.match(r"^[A-Z]\d{9}$", na=False)

    log_issue(
        "DQ-05",
        "Medium",
        "gra_tin",
        "Standardized to uppercase. Created has_valid_tin and gra_tin_pending_flag."
    )

    # DQ-06: MoMo account encoding
    momo_map = {
        "yes": 1, "y": 1, "1": 1, "true": 1,
        "no": 0, "n": 0, "0": 0, "false": 0
    }

    df["has_momo_account_clean"] = (
        df["has_momo_account"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map(momo_map)
    )

    df["has_momo_account_unmapped_flag"] = df["has_momo_account_clean"].isna()

    log_issue(
        "DQ-06",
        "Critical",
        "has_momo_account",
        "Normalized Yes/No/Y/N/1/0/TRUE/FALSE into binary 1/0."
    )

    # DQ-07: Owner gender
    gender_map = {
        "male": "Male",
        "m": "Male",
        "female": "Female",
        "f": "Female"
    }

    df["owner_gender_clean"] = (
        df["owner_gender"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map(gender_map)
    )

    df["owner_gender_unmapped_flag"] = df["owner_gender_clean"].isna()

    log_issue(
        "DQ-07",
        "High",
        "owner_gender",
        "Normalized gender values to Male/Female."
    )

    # Convert numeric columns
    numeric_columns = [
        "monthly_momo_volume_ghs",
        "avg_monthly_bank_balance_ghs",
        "loan_amount_requested_ghs",
        "collateral_value_ghs",
        "owner_age",
        "years_in_operation",
        "credit_bureau_score",
        "previous_loan_count",
        "loan_default",
        "num_employees",
        "bank_account_tenure_months"
    ]

    for col in numeric_columns:
        df[col] = clean_numeric(df[col])

    # DQ-08: Owner age
    df["owner_age_invalid_flag"] = (
        df["owner_age"].lt(18) |
        df["owner_age"].gt(80) |
        df["owner_age"].isna()
    )

    df["owner_age_clean"] = df["owner_age"]
    df.loc[df["owner_age_invalid_flag"], "owner_age_clean"] = np.nan

    log_issue(
        "DQ-08",
        "Critical",
        "owner_age",
        "Ages below 18, above 80, or missing were nulled and flagged."
    )

    # DQ-09: Years in operation
    df["years_in_operation_invalid_flag"] = (
        df["years_in_operation"].lt(0) |
        df["years_in_operation"].gt(50) |
        df["years_in_operation"].isna()
    )

    df["years_in_operation_clean"] = df["years_in_operation"].clip(lower=0, upper=50)
    df.loc[df["years_in_operation_invalid_flag"], "years_in_operation_clean"] = np.nan

    log_issue(
        "DQ-09",
        "High",
        "years_in_operation",
        "Negative values and values above 50 were nulled and flagged."
    )

    # DQ-10: Credit bureau score
        # DQ-10: Credit bureau score
    df["credit_bureau_missing_flag"] = df["credit_bureau_score"].isna()

    df["credit_bureau_below_300_flag"] = df["credit_bureau_score"].lt(300)
    df["credit_bureau_above_850_flag"] = df["credit_bureau_score"].gt(850)

    df["credit_bureau_out_of_range_flag"] = (
        df["credit_bureau_below_300_flag"] |
        df["credit_bureau_above_850_flag"]
    )

    df["credit_bureau_score_clean"] = df["credit_bureau_score"]

    df.loc[
        df["credit_bureau_missing_flag"] |
        df["credit_bureau_out_of_range_flag"],
        "credit_bureau_score_clean"
    ] = np.nan

    # Sentinel value for modelling:
    # -1 means no usable bureau score
    df["credit_bureau_score_model"] = (
        df["credit_bureau_score_clean"].fillna(-1)
    )

    df["credit_bureau_status"] = "Valid bureau score"

    df.loc[
        df["credit_bureau_missing_flag"],
        "credit_bureau_status"
    ] = "No bureau record"

    df.loc[
        df["credit_bureau_below_300_flag"],
        "credit_bureau_status"
    ] = "Invalid below range"

    df.loc[
        df["credit_bureau_above_850_flag"],
        "credit_bureau_status"
    ] = "Invalid above range"

    log_issue(
        "DQ-10",
        "High",
        "credit_bureau_score",
        "Missing bureau history preserved using sentinel value -1 plus missingness and out-of-range flags."
    )


    # DQ-11: MoMo inconsistency
    df["has_momo_from_volume"] = np.where(
        df["monthly_momo_volume_ghs"].fillna(0).gt(0),
        1,
        0
    )

    df["momo_account_volume_conflict_flag"] = (
        df["has_momo_account_clean"].notna() &
        df["has_momo_account_clean"].ne(df["has_momo_from_volume"])
    )

    df["has_momo_final"] = df["has_momo_from_volume"]

    log_issue(
        "DQ-11",
        "High",
        "has_momo_account, monthly_momo_volume_ghs",
        "Created has_momo_from_volume and conflict flag. Transaction evidence used as has_momo_final."
    )

    # DQ-12: Collateral
    df["has_collateral"] = np.where(
        df["collateral_type"].isin(SECURED_COLLATERAL_TYPES),
        1,
        0
    )

    df["collateral_missing_value_flag"] = (
        df["collateral_type"].isin(SECURED_COLLATERAL_TYPES) &
        df["collateral_value_ghs"].isna()
    )

    df["collateral_value_without_collateral_flag"] = (
        df["collateral_type"].isin(UNSECURED_COLLATERAL_TYPES) &
        df["collateral_value_ghs"].fillna(0).gt(0)
    )

    df["collateral_mismatch_flag"] = (
        df["collateral_missing_value_flag"] |
        df["collateral_value_without_collateral_flag"]
    )

    log_issue(
        "DQ-12",
        "High",
        "collateral_type, collateral_value_ghs",
        "Created has_collateral and collateral mismatch flags."
    )

    # DQ-13: Loan-to-revenue ratio
    df["loan_to_revenue_ratio"] = (
        df["loan_amount_requested_ghs"] /
        df["annual_revenue_clean_ghs"]
    )

    df.loc[
        df["annual_revenue_clean_ghs"].isna() |
        df["annual_revenue_clean_ghs"].le(0),
        "loan_to_revenue_ratio"
    ] = np.nan

    df["loan_to_revenue_extreme_flag"] = df["loan_to_revenue_ratio"].gt(5)

    log_issue(
        "DQ-13",
        "Medium",
        "loan_amount_requested_ghs, annual_revenue_ghs",
        "Created loan_to_revenue_ratio and flagged ratios above 5x."
    )

    # DQ-14: Previous default
    df["previous_default_clean"] = (
        df["previous_default"]
        .astype("string")
        .str.strip()
        .str.lower()
        .map({
            "yes": "Yes",
            "no": "No"
        })
    )

    df["previous_default_status"] = "Review"

    df.loc[
        df["previous_loan_count"].eq(0),
        "previous_default_status"
    ] = "No prior loan history"

    df.loc[
        df["previous_loan_count"].gt(0) & df["previous_default_clean"].eq("Yes"),
        "previous_default_status"
    ] = "Prior default"

    df.loc[
        df["previous_loan_count"].gt(0) & df["previous_default_clean"].eq("No"),
        "previous_default_status"
    ] = "No prior default"

    df.loc[
        df["previous_loan_count"].gt(0) & df["previous_default_clean"].isna(),
        "previous_default_status"
    ] = "Missing despite prior loan history"

    df["no_prior_loan_history_flag"] = df["previous_default_status"].eq("No prior loan history")
    df["prior_default_flag"] = df["previous_default_status"].eq("Prior default")

    log_issue(
        "DQ-14",
        "High",
        "previous_default, previous_loan_count",
        "Missing previous_default was not imputed as No. Zero prior loans treated as no prior loan history."
    )

    # Overall review flag
    review_flag_columns = [
        "duplicate_application_id_flag",
        "annual_revenue_usd_flag",
        "application_date_needs_review_flag",
        "phone_leading_zero_added_flag",
        "gra_tin_pending_flag",
        "has_momo_account_unmapped_flag",
        "owner_gender_unmapped_flag",
        "owner_age_invalid_flag",
        "years_in_operation_invalid_flag",
        "credit_bureau_below_300_flag",
        "credit_bureau_above_850_flag",
        "momo_account_volume_conflict_flag",
        "collateral_mismatch_flag",
        "loan_to_revenue_extreme_flag"
    ]

    df["any_data_quality_review_flag"] = (
        df[review_flag_columns]
        .fillna(False)
        .astype(bool)
        .any(axis=1)
    )

    data_quality_log = pd.DataFrame(dq_log).drop_duplicates()

    return df, data_quality_log

    # Run preprocessing
df_clean, data_quality_log = preprocess_applications(df_raw)

print("Cleaned dataset shape:", df_clean.shape)

# Data quality summary
dq_summary = pd.Series({
    "rows_after_cleaning": len(df_clean),
    "duplicate_application_ids_remaining": df_clean["duplicate_application_id_flag"].sum(),
    "annual_revenue_usd_escalate": df_clean["annual_revenue_usd_flag"].sum(),
    "dates_needing_review": df_clean["application_date_needs_review_flag"].sum(),
    "invalid_phone_numbers": (~df_clean["phone_valid_flag"]).sum(),
    "pending_tin": df_clean["gra_tin_pending_flag"].sum(),
    "valid_tin": df_clean["has_valid_tin"].sum(),
    "momo_encoding_unmapped": df_clean["has_momo_account_unmapped_flag"].sum(),
    "owner_gender_unmapped": df_clean["owner_gender_unmapped_flag"].sum(),
    "invalid_owner_age": df_clean["owner_age_invalid_flag"].sum(),
    "invalid_years_in_operation": df_clean["years_in_operation_invalid_flag"].sum(),
    "credit_bureau_missing": df_clean["credit_bureau_missing_flag"].sum(),
    "credit_bureau_below_300": df_clean["credit_bureau_below_300_flag"].sum(),
    "credit_bureau_above_850": df_clean["credit_bureau_above_850_flag"].sum(),
    "momo_conflicts": df_clean["momo_account_volume_conflict_flag"].sum(),
    "collateral_mismatches": df_clean["collateral_mismatch_flag"].sum(),
    "loan_to_revenue_gt_5x": df_clean["loan_to_revenue_extreme_flag"].sum(),
    "records_needing_any_review": df_clean["any_data_quality_review_flag"].sum()
})

print("\nData Quality Summary:")
display(dq_summary)

print("\nData Quality Handling Log:")
display(data_quality_log)

print("\nCleaned Data Preview:")
display(df_clean.head())



Cleaned dataset shape: (3000, 90)

Data Quality Summary:


,0
rows_after_cleaning,3000
duplicate_application_ids_remaining,0
annual_revenue_usd_escalate,151
dates_needing_review,1267
invalid_phone_numbers,0
pending_tin,106
valid_tin,1875
momo_encoding_unmapped,0
owner_gender_unmapped,0
invalid_owner_age,41



Data Quality Handling Log:


,issue_code,severity,field,handling
0,DQ-00,Governance,"ethnic_group, disability_status",Removed from modelling dataset. May be retaine...
1,DQ-01,Critical,application_id,Removed 37 exact duplicate rows. Remaining dup...
2,DQ-02,High,annual_revenue_ghs,Parsed GHS values. Dollar-denominated values f...
3,DQ-03,High,application_date,Parsed likely date formats. Ambiguous or inval...
4,DQ-04,Medium,contact_phone,"Stored as string, removed non-digits, restored..."
5,DQ-05,Medium,gra_tin,Standardized to uppercase. Created has_valid_t...
6,DQ-06,Critical,has_momo_account,Normalized Yes/No/Y/N/1/0/TRUE/FALSE into bina...
7,DQ-07,High,owner_gender,Normalized gender values to Male/Female.
8,DQ-08,Critical,owner_age,"Ages below 18, above 80, or missing were nulle..."
9,DQ-09,High,years_in_operation,Negative values and values above 50 were nulle...



Cleaned Data Preview:


,application_id,business_name,sector,region,years_in_operation,owner_gender,owner_age,num_employees,annual_revenue_ghs,monthly_momo_volume_ghs,...,collateral_missing_value_flag,collateral_value_without_collateral_flag,collateral_mismatch_flag,loan_to_revenue_ratio,loan_to_revenue_extreme_flag,previous_default_clean,previous_default_status,no_prior_loan_history_flag,prior_default_flag,any_data_quality_review_flag
0,SBG-2025-01722,Achimota Farms,Fishing,Ashanti,0.6,Female,34,2,11218.55,<NA>,...,False,False,False,0.992238,False,NaN,No prior loan history,True,False,True
1,SBG-2025-02335,Kofi MotorsCo,Hospitality,Savannah,0.6,Female,31,19,5620.4,6873.49,...,False,False,False,6.259818,True,Yes,Prior default,False,True,True
2,SBG-2025-00444,Dansoman TradingLtd,Hospitality,Greater Accra,2.3,female,32,8,39586.37,3048.94,...,False,False,False,1.172802,False,No,No prior default,False,False,False
3,SBG-2025-02333,Airport City ConsultingCo,Manufacturing,Ashanti,4.6,male,56,5,45732.04,<NA>,...,False,False,False,2.021563,False,No,No prior default,False,False,False
4,SBG-2025-02884,Grace Foods LtdEnterprises,Education,Ashanti,0.4,Male,39,4,48242.71,4302.03,...,True,False,True,0.370605,False,No,No prior default,False,False,True


In [ ]:
df_clean, data_quality_log = preprocess_applications(df_raw)

[col for col in df_clean.columns if "credit_bureau" in col]


['credit_bureau_score',
 'credit_bureau_score_raw',
 'credit_bureau_missing_flag',
 'credit_bureau_below_300_flag',
 'credit_bureau_above_850_flag',
 'credit_bureau_out_of_range_flag',
 'credit_bureau_score_clean',
 'credit_bureau_score_model',
 'credit_bureau_status']

## Model Choice Justification

Two models were trained and compared:

1. Logistic Regression
2. Random Forest Classifier

Both models output probability scores for loan default rather than only binary predictions. Class imbalance was addressed using stratified train/test splitting and class-weighted learning.

Accuracy was not used as the primary evaluation metric because the dataset is imbalanced: most applicants did not default. A model could achieve high accuracy by mostly predicting non-default, while failing to identify risky borrowers.

The models were evaluated using:
- ROC-AUC
- Precision
- Recall
- F1 score
- PR-AUC / Average Precision

PR-AUC is especially important because it focuses on performance for the minority default class.

Logistic Regression is useful because it is interpretable and easier to explain to credit stakeholders. Random Forest is useful because it can capture nonlinear relationships and interactions between borrower characteristics, collateral, bureau information, and financial ratios.

The preferred model should balance strong PR-AUC and recall with explainability. For a bank prototype, a slightly simpler model may be preferred if performance is similar, because credit decisioning requires transparency and governance.

## Why Random Forest Was Selected

Random Forest was selected as the primary model for this prototype because it offered the best balance of predictive performance, implementation simplicity, and governance suitability.

The project considered more advanced gradient-boosting alternatives such as XGBoost and LightGBM. These models often perform very well on structured tabular credit-risk data, especially when there are nonlinear relationships and feature interactions. However, for this prototype, Random Forest was preferred for four reasons:

1. **Strong baseline performance without heavy tuning**  
   Random Forest improved on Logistic Regression in ranking performance, with stronger ROC-AUC and PR-AUC. This made it a better candidate for assigning relative borrower risk.

2. **Lower implementation complexity**  
   Random Forest is available directly in scikit-learn and does not require additional package installation or environment setup. This is useful for a prototype that needs to be reproducible in Google Colab and easy for reviewers to run.

3. **Governance and explainability**  
   Random Forest is less transparent than Logistic Regression, but it is still easier to explain and govern than more complex boosted ensembles. Feature importance, partial dependence, and rule-based overlays can be used to explain broad model behaviour to credit stakeholders.

4. **Reduced overfitting risk for a small dataset**  
   The dataset has only about 3,000 applications. XGBoost and LightGBM can be powerful, but they require careful hyperparameter tuning, validation design, and calibration to avoid overfitting. Random Forest provided a more conservative modelling choice for the current sample size.

Logistic Regression was retained as a benchmark because it is highly interpretable and useful for governance comparison. XGBoost and LightGBM would be reasonable candidates for a later production phase, once the bank has more historical outcomes, a stronger validation framework, and a formal model-risk management process.

Because Random Forest probabilities can be poorly calibrated, the selected Random Forest model was calibrated before being used in the decision engine. Final approve / refer / decline decisions are therefore based on calibrated probability scores plus business rules, not raw model outputs alone.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import pandas as pd
import numpy as np

# Target: historical loan default outcome
target = "loan_default"

# Keep only rows with known target
model_df = df_clean[df_clean[target].notna()].copy()

model_df[target] = model_df[target].astype(int)

# Features selected from cleaned/preprocessed fields
feature_cols = [
    "sector",
    "region",
    "years_in_operation_clean",
    "owner_age_clean",
    "num_employees",
    "annual_revenue_clean_ghs",
    "monthly_momo_volume_ghs",
    "avg_monthly_bank_balance_ghs",
    "bank_account_tenure_months",
    "has_momo_final",
    "has_valid_tin",
    "loan_amount_requested_ghs",
    "loan_purpose",
    "has_collateral",
    "collateral_value_ghs",
    "previous_loan_count",
    "prior_default_flag",
    "no_prior_loan_history_flag",
    "credit_bureau_score_model",
    "credit_bureau_missing_flag",
    "credit_bureau_out_of_range_flag",
    "credit_bureau_status",
    "loan_to_revenue_ratio"
]



X = model_df[feature_cols].copy()
y = model_df[target].copy()

print("Target distribution:")
print(y.value_counts(normalize=True))
print(y.value_counts())

# First split: hold out final test set
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second split: create validation set from remaining data
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.25,
    random_state=42,
    stratify=y_trainval
)

print("Train rows:", X_train.shape[0])
print("Validation rows:", X_val.shape[0])
print("Test rows:", X_test.shape[0])

numeric_features = X.select_dtypes(include=["int64", "float64", "Int64", "Float64", "boolean"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

model_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", model_preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", model_preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=20,
            class_weight="balanced",
            random_state=42
        ))
    ]
)

logistic_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)

print("Models trained.")

def evaluate_model(model, model_name, X_test, y_test, threshold=0.50):
    default_probability = model.predict_proba(X_test)[:, 1]
    predictions = (default_probability >= threshold).astype(int)

    results = {
        "model": model_name,
        "roc_auc": roc_auc_score(y_test, default_probability),
        "pr_auc": average_precision_score(y_test, default_probability),
        "precision": precision_score(y_test, predictions, zero_division=0),
        "recall": recall_score(y_test, predictions, zero_division=0),
        "f1_score": f1_score(y_test, predictions, zero_division=0)
    }

    print(f"\n{model_name}")
    print("-" * 40)
    print("ROC-AUC:", round(results["roc_auc"], 4))
    print("PR-AUC:", round(results["pr_auc"], 4))
    print("Precision:", round(results["precision"], 4))
    print("Recall:", round(results["recall"], 4))
    print("F1 Score:", round(results["f1_score"], 4))
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, predictions))
    print("\nClassification Report:")
    print(classification_report(y_test, predictions, zero_division=0))

    return results, default_probability

log_results, log_probs = evaluate_model(
    logistic_model,
    "Logistic Regression",
    X_test,
    y_test
)

rf_results, rf_probs = evaluate_model(
    random_forest_model,
    "Random Forest",
    X_test,
    y_test
)

comparison = pd.DataFrame([log_results, rf_results])
comparison

from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss

calibrated_rf_model = CalibratedClassifierCV(
    estimator=random_forest_model,
    method="isotonic",
    cv=3
)

calibrated_rf_model.fit(X_train, y_train)

# Validation probabilities for threshold tuning
val_default_probability = calibrated_rf_model.predict_proba(X_val)[:, 1]
val_approval_score = 1 - val_default_probability

def threshold_tradeoff_table(y_true, default_probs):
    rows = []

    for threshold in np.arange(0.05, 0.96, 0.05):
        pred_default = (default_probs >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, pred_default).ravel()

        rows.append({
            "default_threshold": round(threshold, 2),
            "defaults_caught_tp": tp,
            "defaults_missed_fn": fn,
            "good_loans_flagged_fp": fp,
            "good_loans_cleared_tn": tn,
            "precision": precision_score(y_true, pred_default, zero_division=0),
            "recall": recall_score(y_true, pred_default, zero_division=0),
            "f1_score": f1_score(y_true, pred_default, zero_division=0),
            "predicted_default_rate": pred_default.mean()
        })

    return pd.DataFrame(rows)
# Threshold trade-off analysis on validation set
threshold_table_val = threshold_tradeoff_table(
    y_val,
    val_default_probability
)

threshold_table_val

rf_raw_probs = random_forest_model.predict_proba(X_test)[:, 1]
rf_calibrated_probs = calibrated_rf_model.predict_proba(X_test)[:, 1]

calibration_comparison = pd.DataFrame({
    "model": ["Random Forest - Raw", "Random Forest - Calibrated"],
    "roc_auc": [
        roc_auc_score(y_test, rf_raw_probs),
        roc_auc_score(y_test, rf_calibrated_probs)
    ],
    "pr_auc": [
        average_precision_score(y_test, rf_raw_probs),
        average_precision_score(y_test, rf_calibrated_probs)
    ],
    "brier_score": [
        brier_score_loss(y_test, rf_raw_probs),
        brier_score_loss(y_test, rf_calibrated_probs)
    ]
})

calibration_comparison

def threshold_tradeoff_table(y_true, default_probs):
    rows = []

    for threshold in np.arange(0.05, 0.96, 0.05):
        pred_default = (default_probs >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_true, pred_default).ravel()

        rows.append({
            "default_threshold": round(threshold, 2),
            "defaults_caught": tp,
            "defaults_missed": fn,
            "good_loans_flagged": fp,
            "good_loans_cleared": tn,
            "precision": precision_score(y_true, pred_default, zero_division=0),
            "recall": recall_score(y_true, pred_default, zero_division=0),
            "f1_score": f1_score(y_true, pred_default, zero_division=0),
            "predicted_default_rate": pred_default.mean()
        })

    return pd.DataFrame(rows)

threshold_table = threshold_tradeoff_table(
    y_test,
    rf_calibrated_probs
)

threshold_table


scored_test = X_test.copy()
scored_test["actual_default"] = y_test.values
scored_test["logistic_default_probability"] = log_probs
scored_test["random_forest_default_probability"] = rf_probs

scored_test.head()

comparison.sort_values("pr_auc", ascending=False)




Target distribution:
loan_default
0    0.867
1    0.133
Name: proportion, dtype: float64
loan_default
0    2601
1     399
Name: count, dtype: int64
Train rows: 1800
Validation rows: 600
Test rows: 600


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Models trained.

Logistic Regression
----------------------------------------
ROC-AUC: 0.5304
PR-AUC: 0.1369
Precision: 0.1359
Recall: 0.525
F1 Score: 0.2159

Confusion Matrix:
[[253 267]
 [ 38  42]]

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.49      0.62       520
           1       0.14      0.53      0.22        80

    accuracy                           0.49       600
   macro avg       0.50      0.51      0.42       600
weighted avg       0.77      0.49      0.57       600


Random Forest
----------------------------------------
ROC-AUC: 0.5991
PR-AUC: 0.1825
Precision: 0.225
Recall: 0.225
F1 Score: 0.225

Confusion Matrix:
[[458  62]
 [ 62  18]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.88      0.88       520
           1       0.23      0.23      0.23        80

    accuracy                           0.79       600
   macro avg       0.55      0.

,model,roc_auc,pr_auc,precision,recall,f1_score
1,Random Forest,0.599135,0.182502,0.225000,0.225,0.225000
0,Logistic Regression,0.530433,0.136908,0.135922,0.525,0.215938


## Decision Rules

The model produces a probability of default. This was converted into an approval score using:

`approval_score = 1 - default_probability`

The approval score was mapped to recommendations using the following thresholds:

- `0.00 - 0.49`: DECLINE
- `0.50 - 0.79`: REFER TO HUMAN
- `0.80 - 1.00`: APPROVE

Two hard override rules were added:

1. Applicants with prior default history are automatically declined.
2. Applications with unresolved data quality issues are referred to a human unless already declined.

The final recommendation combines model score thresholds with business rules.


In [ ]:
# Use selected calibrated model
selected_model = calibrated_rf_model

# Final test probabilities used only after threshold review
test_default_probability = selected_model.predict_proba(X_test)[:, 1]
test_approval_score = 1 - test_default_probability

decision_test = X_test.copy()
decision_test["actual_default"] = y_test.values
decision_test["calibrated_default_probability"] = test_default_probability
decision_test["approval_score"] = test_approval_score

def score_to_recommendation(approval_score):
    if approval_score < 0.50:
        return "DECLINE"
    elif approval_score < 0.80:
        return "REFER TO HUMAN"
    else:
        return "APPROVE"

decision_test["model_recommendation"] = decision_test["approval_score"].apply(
    score_to_recommendation
)

decision_test["final_recommendation"] = decision_test["model_recommendation"]
decision_test["override_reason"] = ""

# Hard override 1: prior default history = decline
mask_prior_default = decision_test["prior_default_flag"] == True

decision_test.loc[
    mask_prior_default,
    "final_recommendation"
] = "DECLINE"

decision_test.loc[
    mask_prior_default,
    "override_reason"
] = "Hard rule: prior default history"

# Hard override 2: only critical data issues = refer
# Use the full model_df for rule-only flags that were removed from model features
rules_test = model_df.loc[X_test.index].copy()

critical_review_flags = (
    rules_test["annual_revenue_usd_flag"].fillna(False).astype(bool) |
    rules_test["collateral_missing_value_flag"].fillna(False).astype(bool) |
    rules_test["credit_bureau_out_of_range_flag"].fillna(False).astype(bool)
)

mask_critical_data_quality = (
    critical_review_flags &
    (decision_test["final_recommendation"] != "DECLINE")
)

decision_test.loc[
    mask_critical_data_quality,
    "final_recommendation"
] = "REFER TO HUMAN"

decision_test.loc[
    mask_critical_data_quality,
    "override_reason"
] = "Hard rule: critical data issue requires human review"

# Build detailed recommendation output
decision_output = decision_test.copy()

decision_output["application_id"] = model_df.loc[X_test.index, "application_id"].values
decision_output["business_name"] = model_df.loc[X_test.index, "business_name"].values

# Add specific rule-layer flags for transparency
decision_output["annual_revenue_usd_flag"] = model_df.loc[
    X_test.index,
    "annual_revenue_usd_flag"
].values

decision_output["collateral_missing_value_flag"] = model_df.loc[
    X_test.index,
    "collateral_missing_value_flag"
].values

decision_output["credit_bureau_out_of_range_flag"] = model_df.loc[
    X_test.index,
    "credit_bureau_out_of_range_flag"
].values

decision_output = decision_output[
    [
        "application_id",
        "business_name",
        "actual_default",
        "calibrated_default_probability",
        "approval_score",
        "model_recommendation",
        "final_recommendation",
        "override_reason",
        "prior_default_flag",
        "annual_revenue_usd_flag",
        "collateral_missing_value_flag",
        "credit_bureau_out_of_range_flag",
        "loan_to_revenue_ratio",
        "credit_bureau_status",
        "credit_bureau_missing_flag"
    ]
].sort_values(
    by="approval_score",
    ascending=False
)

decision_output.head(50)





,application_id,business_name,actual_default,calibrated_default_probability,approval_score,model_recommendation,final_recommendation,override_reason,prior_default_flag,annual_revenue_usd_flag,collateral_missing_value_flag,credit_bureau_out_of_range_flag,loan_to_revenue_ratio,credit_bureau_status,credit_bureau_missing_flag
1011,SBG-2025-01592,Prestea Gold ServicesCo,0,0.000000,1.000000,APPROVE,APPROVE,,False,False,False,False,0.018098,Valid bureau score,False
858,SBG-2025-00446,Wa LivestockLtd,0,0.021505,0.978495,APPROVE,APPROVE,,False,False,False,<NA>,12.069693,No bureau record,True
1750,SBG-2025-02020,Nana Ama TextilesEnterprises,0,0.021505,0.978495,APPROVE,APPROVE,,False,False,False,False,0.031826,Valid bureau score,False
1144,SBG-2025-02234,Sowutuom VenturesLtd,0,0.021505,0.978495,APPROVE,APPROVE,,False,False,False,<NA>,13.84919,No bureau record,True
2913,SBG-2025-02732,Dzorwulu DigitalCo,0,0.026667,0.973333,APPROVE,APPROVE,,False,False,False,<NA>,0.024879,No bureau record,True
178,SBG-2025-01836,Madina Auto PartsLtd,1,0.026667,0.973333,APPROVE,APPROVE,,False,False,False,False,0.004636,Valid bureau score,False
1772,SBG-2025-00098,Agbogbloshie RecyclingCo,0,0.029457,0.970543,APPROVE,APPROVE,,False,False,False,False,0.059582,Valid bureau score,False
861,SBG-2025-01117,Lapaz CateringEnterprises,0,0.029457,0.970543,APPROVE,APPROVE,,False,False,False,False,0.059014,Valid bureau score,False
1583,SBG-2025-02148,Axim FisheriesCo,0,0.029457,0.970543,APPROVE,APPROVE,,False,False,False,False,0.860498,Valid bureau score,False
2947,SBG-2025-02265,Dzorwulu DigitalGH,0,0.038481,0.961519,APPROVE,APPROVE,,False,False,False,<NA>,<NA>,No bureau record,True


In [ ]:
recommendation_distribution = (
    decision_test["final_recommendation"]
    .value_counts()
    .rename_axis("recommendation")
    .reset_index(name="count")
)

recommendation_distribution["percent"] = (
    recommendation_distribution["count"] /
    recommendation_distribution["count"].sum()
).round(4)

recommendation_distribution


,recommendation,count,percent
0,APPROVE,448,0.7467
1,REFER TO HUMAN,82,0.1367
2,DECLINE,70,0.1167


## Probability Calibration and Threshold Justification

The Random Forest model was selected as the primary scoring model because it had stronger ranking performance than Logistic Regression. However, raw Random Forest probabilities are not always well calibrated. This matters because the decision engine uses probability thresholds to assign applications to APPROVE, REFER TO HUMAN, or DECLINE.

To address this, the Random Forest model was calibrated using `CalibratedClassifierCV` with isotonic calibration. Calibration quality was reviewed using the Brier score, where lower values indicate better probability calibration.

Thresholds were then evaluated using a validation trade-off table. This table shows how different default-probability cutoffs affect:

- defaults caught
- defaults missed
- good loans incorrectly flagged
- precision
- recall
- F1 score
- predicted default rate

The final decision engine uses calibrated default probabilities, converted into an approval score:

`approval_score = 1 - calibrated_default_probability`

The prototype thresholds are:

- `approval_score >= 0.80`: APPROVE
- `0.50 <= approval_score < 0.80`: REFER TO HUMAN
- `approval_score < 0.50`: DECLINE

These thresholds reflect a conservative SME lending risk appetite. Applications with high approval scores are approved automatically. Borderline cases are referred to a human analyst. Low approval scores are declined, especially where additional hard-rule risk factors exist.

Hard override rules are also applied. For example, prior default history leads to decline, and unresolved data quality issues lead to human referral unless the case is already declined.


## Production Architecture Diagram

```mermaid
flowchart LR
    A["SME applicant submits loan request"] --> B["Digital channel / RM portal"]
    B --> C["Document upload: bank statements, invoices, IDs, registration docs"]

    C --> D["Azure AI Document Intelligence + LLM extraction"]
    D --> E["Structured application data"]

    E --> F["Data validation + preprocessing module"]
    F --> G["Feature engineering"]
    G --> H["Credit scoring model<br/>Azure Machine Learning Online Endpoint"]

    H --> I["Probability of default"]
    I --> J["Business rules engine"]

    J --> K{"Final recommendation"}

    K -->|Low risk| L["APPROVE"]
    K -->|High risk or hard decline rule| M["DECLINE"]
    K -->|Uncertain / data quality / policy exception| N["REFER TO HUMAN"]

    N --> O["Credit analyst review"]
    O --> P["Final credit decision"]

    L --> Q["Decision stored"]
    M --> Q
    P --> Q

    Q --> R["Azure SQL / Data Lake"]
    Q --> S["Monitoring dashboard"]

    S --> T["Model monitoring:<br/>drift, default rate, approval rate, overrides"]
    T --> U{"Retrain trigger?"}
    U -->|Yes| V["Azure ML training pipeline"]
    V --> W["Model registry + validation"]
    W --> H




```markdown
## Where the LLM Fits

The LLM should not make the final credit decision. It should support document and text-heavy tasks, such as:

- extracting fields from uploaded bank statements, invoices, business registration documents, IDs, and tax documents
- summarizing relationship manager notes
- identifying missing documents or inconsistencies
- explaining why an application was referred to a human

The LLM output should be validated before entering the scoring model.

## Where the Human Stays in the Loop

A human credit analyst remains in the loop when:

- the model score falls in the referral band
- key data quality flags are present
- collateral value is missing or inconsistent
- annual revenue appears to be in USD or cannot be validated
- the applicant has no bureau record and weak alternative data
- the decision is overridden by policy
- the loan amount exceeds affordability thresholds

The human review decision is stored and used later for monitoring and retraining.

## Azure Deployment

A production version could be deployed on Azure using:

- **Azure App Service or Azure Functions** for the SME application/API layer
- **Azure AI Document Intelligence** for document extraction
- **Azure OpenAI / LLM service** for document reasoning, summarization, and explanation support
- **Azure Machine Learning** for model training, model registry, and deployment
- **Azure ML Managed Online Endpoint** for real-time scoring
- **Azure SQL Database or Azure Data Lake Storage** for applications, decisions, and monitoring data
- **Azure Key Vault** for secrets and credentials
- **Azure Monitor / Application Insights** for logging, latency, errors, and service health
- **Power BI** for credit-risk and portfolio monitoring dashboards

Azure Machine Learning online endpoints are designed for real-time inference through a stable endpoint URL, while Azure ML model monitoring can track drift and production model behavior.

## Retrain Triggers

The model should be retrained when one or more of the following occur:

- data drift is detected in key features such as revenue, loan amount, sector, bureau score, or MoMo volume
- model performance falls below an agreed threshold, such as ROC-AUC, PR-AUC, recall, or bad-rate separation
- actual default rates differ materially from predicted default probabilities
- approval, decline, or referral rates shift unexpectedly
- human override rates become high
- new repayment/default outcomes become available
- credit policy changes
- macroeconomic conditions change materially
- a scheduled retraining window occurs, for example quarterly


## Fairness Review and Non-Discrimination

The model was reviewed for features that may create direct discrimination risk, proxy discrimination risk, or fairness concerns in SME lending.

### Direct Protected or Sensitive Features

The following fields were removed from the modelling dataset:

- `ethnic_group`
- `disability_status`

These variables directly identify protected or sensitive characteristics. They were excluded from model training and inference because they should not influence automated approve / decline / refer recommendations.

### Grey-Zone Features Kept With Caution

Some features were retained but require fairness monitoring:

- `owner_gender`
- `owner_age`
- `region`

These variables may have legitimate credit-risk relevance in some contexts, but they can also create fairness risk.

`owner_gender` may reveal gender-based disparities in access to credit.  
`owner_age` may be related to business experience or financial stability, but it can also create age-based adverse impact.  
`region` may capture market conditions, infrastructure, or local economic differences, but it can also act as a proxy for ethnicity, income, or rural/urban access.

For this prototype, these variables are not treated as automatically prohibited, but they should be subject to fairness testing and governance review before production deployment.

### What Was Done

The preprocessing module removes direct sensitive variables from the modelling dataset. It also avoids using identifiers such as `business_name`, `contact_phone`, raw `gra_tin`, `rm_recommendation`, `internal_risk_grade`, and `days_past_due_current` as modelling features.

For grey-zone variables, the recommended treatment is:

- retain only if there is documented business justification
- test for disparate impact before deployment
- compare model performance and recommendation rates with and without the feature
- monitor post-deployment outcomes by protected or vulnerable group
- keep human review available for borderline and exception cases

### Demonstrating Non-Discrimination To The Bank Of Ghana

To demonstrate non-discrimination, the bank should maintain a model governance pack containing:

1. **Feature inventory**
   - list all model features
   - identify protected, sensitive, and proxy-sensitive fields
   - document which features were removed and why
   - justify any grey-zone features retained in the model

2. **Fairness metrics**
   - approval rate by group
   - decline rate by group
   - referral rate by group
   - default rate by group
   - false positive rate and false negative rate by group
   - disparate impact ratio across groups

3. **Benchmark model comparison**
   - train/evaluate one model with grey-zone variables
   - train/evaluate another model without them
   - compare predictive performance and fairness metrics
   - retain grey-zone features only if the performance benefit is material and fairness impact is acceptable

4. **Human oversight**
   - keep human review for borderline applications
   - allow escalation or appeal for declined applications
   - document override reasons
   - review whether overrides are applied consistently across groups

5. **Ongoing monitoring**
   - monitor approval, decline, referral, default, and override rates by segment
   - monitor drift in region, gender, and age distributions
   - retrain or recalibrate if fairness metrics deteriorate
   - report fairness outcomes to model governance and compliance stakeholders

